# Neural Fortran Examples: Merging Two Networks

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/merge_networks.f90`. Two networks read different inputs. Their outputs are concatenated and passed to a third network. The backward pass starts on the third network, and the piece of its input gradient that belongs to each branch is handed back with the `gradient` argument of `backward`.

## Three networks, one sample

```{literalinclude} Fortran_Code/Section_Examples_Merge/app/merge_networks.f90
:language: fortran
:lines: 6-38
```

The sample is fixed.

$$
x^{(1)} = (0.1,\ 0.3,\ 0.5), \qquad x^{(2)} = (0.2,\ 0.4),
$$

$$
y = (0.123456,\ 0.246802,\ 0.369258,\ 0.482604,\ 0.505050,\ 0.628406,\ 0.741852).
$$

`net1` maps 3 inputs through widths 2, 3, and 2, so its output length is 2. `net2` maps 2 inputs through widths 5 and 3, so its output length is 3. `net3` takes an input of length $2+3 = 5$ and a dense layer of width 7, which matches `y`. Every dense layer uses the default sigmoid. The output sizes are read from the last layer's `layer_shape` with `product`, so the third constructor follows from the first two.


## Forward, split the gradient, update

```{literalinclude} Fortran_Code/Section_Examples_Merge/app/merge_networks.f90
:language: fortran
:lines: 40-69
```

Each iteration runs `forward` on the two branches, reads their outputs with `get_output`, and feeds the concatenation `[y1, y2]` to `net3`.

`net3 % backward(y)` uses the quadratic loss against the length-7 target and fills the gradient of every `net3` layer. The dense layer is `layers(2)`. Its `gradient` array has one entry per input of that layer. The first `net1_output_size` entries belong to `net1`, and the rest belong to `net2`. Those slices are passed as the optional `gradient` argument. When that argument is present, `backward` does not recompute an output loss. It starts from the supplied gradient. The `y` argument is still required by the interface. The branch update is driven by the slice.

All three networks then take one SGD step with learning rate 1. The printed number is a real root-mean-square error, unlike the quadratic example, which prints a squared error under an RMSE label:

$$
\mathrm{RMSE} = \sqrt{\frac{1}{N}\sum_{k=1}^{N}\left(\hat y_k - y_k\right)^2}.
$$

`N` is 7. The prediction inside the square root is a fresh pass: `net3 % predict` of the concatenation of `net1 % predict(x1)` and `net2 % predict(x2)`.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Merge/app/merge_networks.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Merge/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `merge_networks` from the release build. Standard error was empty and the process exited 0. The network starts from random weights, so the path belongs to this run. The final error is small because the sample is a single fixed pair.

```{literalinclude} Fortran_Code/Section_Examples_Merge/merge_networks.out
:language: text
```

## What the run did

![RMSE of the merged network](images/nf_merge.png)

At iteration 50 the RMSE is $5.13 \times 10^{-3}$. It falls by about a factor of six every 50 steps through iteration 350, reaches $8.11 \times 10^{-8}$ at iteration 400, and ends at $6.05 \times 10^{-8}$ at iteration 500. The last two steps are no longer a straight line on the log plot. The outputs are already far below the six decimals of the target values.

## Conclusion

The example is how neural-fortran expects two networks to be joined. Concatenate the forward outputs, run `backward` on the network that sees the loss, and pass slices of that network's input gradient into the branches. `select type` is what makes the slice available, because `gradient` lives on `dense_layer`.